# Finance Data Backtesting Demo with cppyy/CppJit

A backtest of a simple linear trading strategy on historical price data.

## The message

 * Mixing Python steering and C++ analysis kernel code gives you performance that can usually not be reached with an implementation that uses higher-level Python libraries like NumPy, Pandas, and scikit-learn.
 * Mixing the two languages is not as hard as it might seem, thanks to the **dynamic bindings**, **seamingless usage of C++ types**, and eventual **helpers for automatic C++ code generation**.
 
## What it does
 
  * The demo trains a linear model on one month of price data and uses it to predict the next month's moves.
  * Compares the resulting strategy (only holding the asset on months with a positive predicted return) against a simple buy-and-hold benchmark.

In [ ]:
import ROOT
import cppyy

from python_to_cpp_helpers import declare_cpp_struct, jit_cpp_code

std = cppyy.gbl.std

import time
import numpy as np
import matplotlib.pyplot as plt

ROOT.gInterpreter;

Include C++ side helpers:

In [ ]:
%%cpp

#include "RTimeSeries.h"

Timing helper:

In [ ]:
def print_time(counter, message):
    end = time.perf_counter()
    t = 1000 * (end - counter)  # in ms
    print(message, "{0:.3f} ms".format(t))
    return end

Declare a C++ data class for our analysis, using Python that gets translated under the hood:

In [ ]:
@declare_cpp_struct
class DataStruct:
    open = 0.0
    close = 0.0
    low = 0.0
    high = 0.0
    volume = 0.0
    date = std.time_t()

Implement analysis logic:
  1. Build feature matrix
  2. Iterate over all months:
       * Fit linear regression on all months so far
       * Evaluate linear regression to predict next month returns

In [ ]:
%%cpp

void process(DataStruct::SoA &data, std::vector<double> &all_preds)
{
   std::vector<double> coef(InputTransformer::nOut, 0.0);
   std::vector<double> intercept(1, 0.0);

   bool last_model = false;
   size_t start_idx = 0;
   auto current_month = time_to_period(data.date[0]);

   for (size_t i = 0; i < data.date.size(); ++i) {
      auto month = time_to_period(data.date[i]);

      if (!(month != current_month || i == data.date.size() - 1))
         continue;

      size_t end_idx = (month != current_month) ? i : i + 1;
      size_t n_rows = end_idx - start_idx;
      size_t n_features = 5; // Open, Close, Low, High, Volume

      // Build X matrix
      std::vector<double> X(n_rows * n_features);
      for (size_t r = 0; r < n_rows; ++r) {
         X[r * n_features + 0] = data.open[start_idx + r];
         X[r * n_features + 1] = data.close[start_idx + r];
         X[r * n_features + 2] = data.low[start_idx + r];
         X[r * n_features + 3] = data.high[start_idx + r];
         X[r * n_features + 4] = data.volume[start_idx + r];
      }

      // Build y vector (Close - Open, skip first element)
      std::vector<double> y(n_rows - 1);
      for (size_t r = 1; r < n_rows; ++r) {
         y[r - 1] = data.close[start_idx + r] - data.open[start_idx + r];
      }

      std::vector<double> y_pred(n_rows);

      if (last_model) {
         // From our analysis library
         evaluateLinearModel(y_pred.data(), 1, n_features, n_rows - 1, X.data(), coef.data(), intercept[0]);
      }

      all_preds.insert(all_preds.end(), y_pred.begin(), y_pred.end());

      fitLinearModel(n_features, n_rows - 1, X.data(), y.data(), coef.data(), intercept.data());

      last_model = true;
      current_month = month;
      start_idx = i;
   }

   // all_preds now contains all predicted values
}

Function for cumulative sums of the predictions. Can be written in Python and can be directory mapped to C++, since it is relatively simple code.

In [ ]:
@jit_cpp_code
def calc_cumsum(
    n: std.size_t,
    cumsum: std.span["double"],
    cumsum2: std.span["double"],
    open: std.span["const double"],
    close: std.span["const double"],
    preds: std.span["const double"],
) -> None:
    cumsum[0] = close[0] - open[0]
    cumsum2[0] = close[0] - open[0] if preds[0] > 0.0 else 0.0
    for i in range(n):
        diff = close[i] - open[i]
        cumsum[i] = cumsum[i - 1] + diff
        cumsum2[i] = cumsum2[i - 1] + (diff if preds[i] > 0.0 else 0.0)

Load the data, run the backtesting analysis.

In [ ]:

data = ROOT.read_csv["DataStruct::SoA"]("data/historical_quotes.csv", delimiter=",", skiprows=1)

# Sort by date: oldest to newest
ROOT.apply_permutation(
    ROOT.make_sort_permutation(data.date), data.open, data.close, data.low, data.high, data.volume, data.date
)

counter = time.perf_counter()

all_preds = std.vector["double"]()

ROOT.process(data, all_preds)

counter = print_time(counter, "Analyzing")

n = data.open.size()

# Compute cumulative sum
cumsum = std.vector["double"](n)
cumsum2 = std.vector["double"](n)

calc_cumsum(n, cumsum, cumsum2, data.open, data.close, all_preds)

Make a plot with daily returns

The equivalent runtimes in a NumPy+Pandas+sklearn-based implementation:
```txt
Analyzing 184.493 ms
```

In [ ]:
plt.plot(cumsum, label="Diff cumsum")
plt.plot(cumsum2, label="Strategy cumsum")
plt.xlabel("Month")
plt.ylabel("Cumulative return")
plt.legend()